# Clase 11 – Práctica: modelos generativos modernos

## Qué vas a practicar

En esta práctica trabajas dos piezas centrales del autoencoder variacional (VAE, por sus siglas en inglés, *Variational Autoencoder*):

- El truco de reparametrización, el mecanismo que permite entrenar el VAE con descenso de gradiente pese al muestreo aleatorio de su espacio latente.
- La interpolación en el espacio latente, la propiedad que convierte ese espacio en un mapa continuo donde moverte de un punto a otro produce transiciones coherentes entre dígitos.

Completa el código donde encuentres un comentario `# TODO`. Ejecuta las celdas en orden: el entrenamiento del ejercicio 1 tarda pocos minutos con GPU en Google Colab, y el modelo resultante lo reutilizas en el ejercicio 2.

## Requisito previo

Haber visto la Clase 11 ("Modelos generativos modernos: autoencoders variacionales") y tener claro el papel de `mu`, `log_var` y la divergencia de Kullback-Leibler (KL) en el objetivo de entrenamiento del VAE.

**Activa una GPU antes de empezar:** en el menú de Colab, ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU.

## Ejercicio 1 — Completa el truco de reparametrización

### Contexto

Imagina que tu equipo quiere generar variaciones sintéticas de imágenes para ampliar un conjunto de datos de entrenamiento, un caso de uso del VAE en salud y biotecnología que viste en la Clase 11. Para eso necesitas primero un VAE que funcione, y el paso que lo diferencia de un autoencoder normal es el muestreo dentro del espacio latente.

El codificador no produce un único punto `z` para cada imagen: produce los parámetros de una distribución gaussiana, la media `mu` y el logaritmo de la varianza `log_var`. Muestrear `z` directamente de esa distribución bloquearía la propagación del gradiente, porque no existe una derivada bien definida de "un número elegido al azar" respecto a los parámetros que definen ese azar.

La solución es el truco de reparametrización: separas el azar (un ruido `epsilon` que no depende de ningún parámetro entrenable) de la transformación determinista y diferenciable que lo combina con `mu` y `sigma`:

z = mu + sigma * epsilon, con epsilon ~ N(0, I)

Recuerda que la red predice `log_var` (el logaritmo de la varianza), no `sigma` directamente, así que necesitas recuperar `sigma` antes de aplicar la fórmula.

### Qué debes completar

En la clase `VAE`, el codificador (`encoder_hidden`, `fc_mu`, `fc_logvar`) y el decodificador (`decoder_hidden`, `decoder_out`) ya están escritos. Completa el método `reparametrizar`:

1. Calcula `sigma` a partir de `log_var`
2. Genera el ruido `epsilon` con la misma forma que `sigma`, muestreado de una N(0, 1)
3. Combina `mu`, `sigma` y `epsilon` según la fórmula de reparametrización

Pista: `torch.exp` convierte log-varianza en varianza (recuerda el factor 0.5 para pasar a desviación típica), y `torch.randn_like` genera ruido con la misma forma y en el mismo dispositivo (CPU o GPU) que el tensor que le pases.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# --- Datos: MNIST, imágenes de 28x28 en escala de grises, valores en [0, 1] ---
transform = transforms.ToTensor()
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

LATENT_DIM = 20  # dimensión del espacio latente z


class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=LATENT_DIM):
        super().__init__()
        # Codificador: de la imagen aplanada a una representación oculta compartida
        self.encoder_hidden = nn.Linear(input_dim, hidden_dim)
        # Dos cabezas separadas: una para la media, otra para el log de la varianza
        # (se predice el logaritmo de la varianza, no la varianza directamente,
        # para que la red pueda producir cualquier número real y aun así obtengamos
        # una varianza siempre positiva al exponenciar)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # Decodificador: del espacio latente de vuelta a los píxeles
        self.decoder_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.encoder_hidden(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparametrizar(self, mu, log_var):
        # TODO: implementa el truco de reparametrización z = mu + sigma * epsilon
        # 1. Recupera sigma a partir de log_var (log_var = log(sigma^2))
        sigma = ...
        # 2. Genera epsilon ~ N(0, I) con la misma forma y dispositivo que sigma
        epsilon = ...
        # 3. Combina mu, sigma y epsilon en el punto latente z
        z = ...
        return z

    def decode(self, z):
        h = F.relu(self.decoder_hidden(z))
        # Sigmoid porque el decodificador modela una Bernoulli por píxel (valores en [0, 1])
        return torch.sigmoid(self.decoder_out(h))

    def forward(self, x):
        mu, log_var = self.encode(x.view(x.size(0), -1))
        z = self.reparametrizar(mu, log_var)
        x_reconstruida = self.decode(z)
        return x_reconstruida, mu, log_var

In [ ]:
def elbo_loss(x_reconstruida, x, mu, log_var):
    # Término de reconstrucción: log-verosimilitud de una Bernoulli, sumada sobre píxeles
    reconstruccion = F.binary_cross_entropy(
        x_reconstruida, x.view(x.size(0), -1), reduction="sum"
    )
    # KL entre q(z|x) = N(mu, sigma^2) y el prior p(z) = N(0, I), forma cerrada
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
    return reconstruccion + kl  # minimizar esto equivale a maximizar el ELBO


modelo = VAE().to(device)
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

EPOCHS = 8
for epoch in range(1, EPOCHS + 1):
    modelo.train()
    perdida_total = 0.0
    for x, _ in train_loader:  # las etiquetas no se usan: el VAE es no supervisado
        x = x.to(device)
        optimizador.zero_grad()
        x_reconstruida, mu, log_var = modelo(x)
        perdida = elbo_loss(x_reconstruida, x, mu, log_var)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item()

    print(f"Época {epoch}/{EPOCHS} — pérdida media: {perdida_total / len(train_data):.2f}")

# --- Comprobación rápida: reconstrucciones frente a originales ---
modelo.eval()
with torch.no_grad():
    x_test, _ = next(iter(test_loader))
    x_test = x_test.to(device)
    x_reconstruida, _, _ = modelo(x_test)

n = 8
fig, ejes = plt.subplots(2, n, figsize=(n * 1.2, 3))
for i in range(n):
    ejes[0, i].imshow(x_test[i].cpu().squeeze(), cmap="gray")
    ejes[0, i].axis("off")
    ejes[1, i].imshow(x_reconstruida[i].cpu().view(28, 28), cmap="gray")
    ejes[1, i].axis("off")
plt.suptitle("Fila superior: dígitos originales — fila inferior: reconstrucciones del VAE")
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
def reparametrizar(self, mu, log_var):
    # 1. sigma se recupera de log_var: log_var = log(sigma^2), así que
    #    sigma = exp(0.5 * log_var). Usar el logaritmo de la varianza en vez de
    #    la varianza directa evita restringir la salida de la red a valores positivos.
    sigma = torch.exp(0.5 * log_var)

    # 2. epsilon es el ruido externo y fijo, independiente de los parámetros
    #    entrenables: aquí es donde vive el azar, aislado del resto del cálculo.
    epsilon = torch.randn_like(sigma)

    # 3. La transformación z = mu + sigma * epsilon es determinista y diferenciable
    #    respecto a mu y sigma, así que el gradiente puede atravesarla sin problema.
    z = mu + sigma * epsilon
    return z
```
</details>

## Ejercicio 2 — Interpola entre dos dígitos en el espacio latente

### Contexto

Ya tienes un VAE entrenado en el ejercicio 1 (la variable `modelo`). Ahora comprueba la propiedad que distingue a un VAE de un autoencoder normal: el espacio latente es un mapa continuo, sin descampados, así que moverte de un punto a otro produce una secuencia de imágenes con sentido, no ruido.

Vas a construir una función `interpolar` que:

1. Codifica dos imágenes de dígitos distintos y se queda con sus medias `mu_a` y `mu_b` (usa la media, no una muestra, para tener un punto de referencia estable de cada dígito)
2. Genera `n_pasos` puntos intermedios entre `mu_a` y `mu_b` mediante interpolación lineal: `z_t = (1 - t) * mu_a + t * mu_b`, con `t` recorriendo valores entre cero y uno
3. Decodifica cada punto intermedio con `modelo.decode(...)`
4. Muestra la secuencia completa en una fila con matplotlib, del dígito `a` al dígito `b`

### Qué debes completar

El andamiaje ya trae la función `interpolar` con la codificación, la decodificación y la visualización escritas. Completa únicamente el cálculo de los `n_pasos` puntos intermedios entre `mu_a` y `mu_b`.

Pista: `torch.linspace(0, 1, n_pasos)` te da los valores de `t`; recuerda moverlos al mismo `device` que `mu_a` y `mu_b`, y darles la forma adecuada para que el broadcasting combine correctamente cada `t` con el vector latente completo.

In [ ]:
def obtener_par_de_digitos(digito_a, digito_b, dataset):
    """Devuelve una imagen de test para cada dígito solicitado (0-9)."""
    imagen_a = next(x for x, etiqueta in dataset if etiqueta == digito_a)
    imagen_b = next(x for x, etiqueta in dataset if etiqueta == digito_b)
    return imagen_a, imagen_b


def interpolar(modelo, digito_a, digito_b, n_pasos=10):
    modelo.eval()
    with torch.no_grad():
        imagen_a, imagen_b = obtener_par_de_digitos(digito_a, digito_b, test_data)
        x_a = imagen_a.view(1, -1).to(device)
        x_b = imagen_b.view(1, -1).to(device)

        # Codifica ambas imágenes y usa la media como punto de referencia en el espacio latente
        mu_a, _ = modelo.encode(x_a)
        mu_b, _ = modelo.encode(x_b)

        # TODO: genera n_pasos valores de t entre 0 y 1 (torch.linspace) y calcula
        # los puntos intermedios z_t = (1 - t) * mu_a + t * mu_b para cada valor de t.
        # mu_a y mu_b tienen forma (1, latent_dim); da a los valores de t la forma
        # (n_pasos, 1) para que el broadcasting produzca un tensor (n_pasos, latent_dim).
        pasos_t = ...
        puntos_intermedios = ...

        imagenes_generadas = modelo.decode(puntos_intermedios).cpu()

    fig, ejes = plt.subplots(1, n_pasos, figsize=(n_pasos * 1.2, 1.6))
    for i in range(n_pasos):
        ejes[i].imshow(imagenes_generadas[i].view(28, 28), cmap="gray")
        ejes[i].axis("off")
    plt.suptitle(f"Interpolación en el espacio latente: {digito_a} → {digito_b}")
    plt.show()


interpolar(modelo, digito_a=3, digito_b=8, n_pasos=10)

<details>
<summary>💡 Ver solución</summary>

```python
# TODO resuelto dentro de interpolar(...):

# n_pasos valores de t, repartidos uniformemente entre 0 y 1, en el mismo
# dispositivo que mu_a y mu_b, con forma (n_pasos, 1) para poder combinarlos
# por broadcasting con vectores latentes de forma (1, latent_dim)
pasos_t = torch.linspace(0, 1, n_pasos, device=device).unsqueeze(1)

# Interpolación lineal: en t=0 obtienes mu_a, en t=1 obtienes mu_b, y en los
# valores intermedios, una combinación ponderada de ambos. El resultado es
# un tensor de forma (n_pasos, latent_dim), un punto latente por paso.
puntos_intermedios = (1 - pasos_t) * mu_a + pasos_t * mu_b
```
</details>

## Cierre

Completaste el truco de reparametrización y comprobaste con tus propios ojos que el espacio latente de un autoencoder variacional es un mapa continuo: la transición entre dos dígitos es gradual, no un salto brusco. Esa misma propiedad es la que hace útil al VAE en biotecnología, diagnóstico por imagen o creación de contenido, en cualquier caso donde necesites moverte de forma controlada entre dos puntos con significado.

Como siguiente paso, prueba a cambiar `LATENT_DIM` a dos dimensiones, entrena de nuevo y dibuja el espacio latente completo con `plt.scatter`, coloreando cada punto según su etiqueta. Verás si los dígitos parecidos —como el cuatro y el nueve— quedan cerca entre sí en el mapa.

**Vuelve al foro de la Clase 11 y comparte tu interpolación entre dos dígitos: cuáles elegiste y qué observaste en la transición.**